### 1. Data Loading and Data Engineering



In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_squared_error


# Load the data
df = pd.read_csv('../survey_results_public.csv')



C:\Users\ReDI\AppData\Local\Temp\ipykernel_13220\3778189721.py:16: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../survey_results_public.csv')


In [136]:
#Select only developers by profession and also in DACH region
df = df[(df['ConvertedCompYearly'] > 0) &
        (df['Country'].isin(['Germany', 'Austria', 'Switzerland'])) &
        (df['MainBranch'] == 'I am a developer by profession')
         ]
print("Data shape:", df.shape)

Data shape: (2343, 172)


***
### 2. Data Splitting

Split the processed data into training and testing sets.

In [137]:
# Define the target variable (y) and features (X)
X = df.drop('ConvertedCompYearly', axis=1)
y = df['ConvertedCompYearly']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (1874, 171)
Shape of X_test: (469, 171)


***
### 3. Data Preprocessing Pipeline

Fill in missing values using ColumnTransformer

In [138]:
# Features to impute with the median (numerical)
numerical_features = ['JobSat']

# Features to impute with the mode and One-Hot Encode (categorical/ordinal)
categorical_features = ['Age','EdLevel', 'DevType', 'RemoteWork','Country']

# A. Numerical Pipeline: Impute with Median
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

# B. Categorical Pipeline: Impute with Mode, then One-Hot Encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')) 
])
 
# C. Combine all steps using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop' # Drop all other columns
)

#disabling this part so we can experiment on other models
#pipeline = Pipeline ([
#    ('preprocessor',preprocessor),
#    ('model', LinearRegression())
#])


# run the preprocessor on X_train and X_test
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

X_train_df = pd.DataFrame(X_train.toarray(), columns=preprocessor.get_feature_names_out())
X_test_df = pd.DataFrame(X_test.toarray(), columns=preprocessor.get_feature_names_out())
X_train_df




Shape of X_train: (1874, 56)
Shape of X_test: (469, 56)


,num__JobSat,cat__Age_18-24 years old,cat__Age_25-34 years old,cat__Age_35-44 years old,cat__Age_45-54 years old,cat__Age_55-64 years old,cat__Age_65 years or older,cat__Age_Prefer not to say,"cat__EdLevel_Associate degree (A.A., A.S., etc.)","cat__EdLevel_Bachelor’s degree (B.A., B.S., B.Eng., etc.)",...,cat__DevType_System administrator,"cat__DevType_UX, Research Ops or UI design professional","cat__RemoteWork_Hybrid (some in-person, leans heavy to flexibility)","cat__RemoteWork_Hybrid (some remote, leans heavy to in-person)",cat__RemoteWork_In-person,cat__RemoteWork_Remote,"cat__RemoteWork_Your choice (very flexible, you can come in when you want or just as needed)",cat__Country_Austria,cat__Country_Germany,cat__Country_Switzerland
0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,7.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,8.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,10.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1869,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1870,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1871,4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1872,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


***
### 4. Plotting Functions

Functions to help visualize the model's performance and structure.

***
### 5. Model Training and Evaluation


In [139]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)


Mean Absolute Error:  26473.751665526466
Mean Squared Error:  1913171197.2579577
Root Mean Squared Error:  43739.81249683128
r2 score: 0.32423994341178375


In [140]:
# Train the model
model = Ridge()
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)

Mean Absolute Error:  26419.083770987076
Mean Squared Error:  1902999927.2838686
Root Mean Squared Error:  43623.387388920964
r2 score: 0.32783258477242927


In [147]:
# Train the model
model = DecisionTreeRegressor(max_depth=1)
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)

Mean Absolute Error:  31999.981028797123
Mean Squared Error:  2369198425.573825
Root Mean Squared Error:  48674.41243172664
r2 score: 0.16316445468700402


In [151]:
# Train the model
model = Lasso(alpha=0.1, max_iter=100000)
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)

Mean Absolute Error:  26473.412579215055
Mean Squared Error:  1913139720.639626
Root Mean Squared Error:  43739.45267878447
r2 score: 0.3242510614138818
